# Semana 3 - Día 1

Bienvenido a Google Colab: computación gratuita y potente en la nube.

Google Colab te da una ventana de tipo notebook en el navegador, conectada a una máquina remota. Ejecutas el código "localmente" en esa máquina.

## Ventajas de Colab

1. ¡Acceso gratuito a GPUs T4!
2. Fácil de compartir código y colaborar
3. Todos usan exactamente el mismo código, sin diferencias de entorno

## Desventajas de Colab

1. Como es gratuito, Google puede desconectarte de la máquina en cualquier momento ("reset the runtime"), y esto pasa más rápido si no hay código ejecutándose. También pueden cambiarte de una T4 a una máquina solo con CPU. A veces pasa en silencio. Tienes que volver a empezar todo desde arriba. Los planes de pago duran más.
2. Tienes que instalar paquetes con pip cada vez (pero hoy no hay instalaciones de pip)
3. Hay cierta latencia: no es tan interactivo como programar en tu propia máquina

## Guía de supervivencia

1. Empieza siempre pulsando la flecha desplegable junto a Connect, arriba a la derecha, y elige "Connect to a hosted runtime: T4"
2. Desde ese menú, "View Resources" para comprobar que tienes GPU y monitorear la memoria
3. Si algo sale mal, Runtime >> Disconnect and Delete Runtime, vuelve a conectar y ejecuta las celdas desde arriba
4. ¡Ejecuta siempre las instalaciones de pip! Ignora los errores de dependencias de pip.

Runtime >> Restart session: reinicia el kernel de Python, pero los paquetes de pip siguen instalados y el disco no cambia.

Runtime >> Disconnect and delete session. Esto borra todo.

In [ ]:
# Tuve que agregar esta instalación de pip desde que grabé el video, ¡porque Hugging Face hizo un cambio incompatible!

!pip install -q --upgrade transformers==4.56.2 diffusers==0.32.2

In [ ]:
# Revisemos la GPU - debería ser una Tesla T4

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('No conectado a una GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Éxito - Conectado a una T4")
  else:
    print("NO CONECTADO A UNA T4")

# Conectar Hugging Face

Necesitarás iniciar sesión en el hub de Hugging Face si todavía no lo has hecho.

1. Si aún no tienes una cuenta, crea una **gratuita** en https://huggingface.co y ve a Settings desde el menú de usuario, arriba a la derecha. Luego selecciona Access Tokens y haz clic en el botón Create new token, cerca de la parte superior derecha. LUEGO asegúrate de hacer clic en el botón "Write" en 'Token type'.

  **IMPORTANTE:** al crear tu token de acceso de Hugging Face, selecciona permisos de WRITE haciendo clic en el botón "Write". Si no, podrías tener problemas después. No "fine-grained" sino "Write".

  En el campo "Token name" puedes poner lo que quieras, como "Mi token HF". Ese campo es solo de referencia.

  Luego pulsa "Create token", cópialo al portapapeles y guarda un registro de tu token. Debe empezar con "hf_..."

2. De vuelta en Colab, pulsa el ícono de "llave" en el panel lateral izquierdo y agrega un nuevo secreto:
  En el campo name escribe `HF_TOKEN`
  En el campo value escribe tu token de acceso real: `hf_...`
  Asegúrate de que el interruptor de acceso del notebook esté ACTIVADO.

3. Ejecuta la celda de abajo para iniciar sesión. Tendrás que hacerlo en cada uno de tus colabs. Es una forma muy útil de manejar tus secretos sin tener que escribirlos en Colab.

In [ ]:
from huggingface_hub import login
from google.colab import userdata


hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)


In [ ]:
# Reinicia el kernel

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
from IPython.display import display
from diffusers import DiffusionPipeline
import torch

pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
pipe.to("cuda")

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = pipe(prompt=prompt, num_inference_steps=30).images[0]

display(image)


In [ ]:
# Reinicia el kernel

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
from diffusers import DiffusionPipeline
import torch

base = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
base.to("cuda")
refiner = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-refiner-1.0", text_encoder_2=base.text_encoder_2, vae=base.vae, torch_dtype=torch.float16, use_safetensors=True, variant="fp16",)
refiner.to("cuda")

# Define cuántos pasos y qué % de pasos ejecuta cada experto (80/20) aquí
n_steps = 40
high_noise_frac = 0.8

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

# ejecuta ambos expertos
image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images

image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]

display(image)

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
!pip install --upgrade datasets==3.6.0

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Estas celdas solo funcionarán en una GPU potente como una A100

No están disponibles en una T4 gratuita.

Solo quiero mostrar lo que es posible con un pequeño presupuesto de pago.

Precios aproximados:

- $9.99 = 100 unidades de cómputo
- Una A100 = 5.37 unidades de cómputo por hora a octubre de 2025 (según mi cuenta)

In [ ]:
# Revisemos la GPU - debería ser una A100

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('No conectado a una GPU')
else:
  print(gpu_info)
  if gpu_info.find('A100') >= 0:
    print("Éxito - Conectado a una NVIDIA A100")
  else:
    print("NO CONECTADO A UNA A100")

## Paso extra rápido: solicitar acceso al modelo

Visita esta página e inicia sesión con tu cuenta de Hugging Face:
https://huggingface.co/black-forest-labs/FLUX.1-schnell

Cerca de la parte superior verás una sección donde puedes solicitar acceso a este modelo restringido. Quizá solo necesites aceptar los términos de servicio. Cuando se apruebe tu acceso, debería cambiar a:

"You have been granted access to this model"

Entonces la siguiente celda debería funcionar. (Si no, verás un error de que no tienes acceso al modelo restringido.)

¡Gracias a Chris C. por la pista!

In [ ]:
import torch
from diffusers import FluxPipeline
from IPython.display import display
from datetime import datetime

start = datetime.now()

pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16).to("cuda")
generator = torch.Generator(device="cuda").manual_seed(0)
prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

# Genera la imagen usando la GPU
image = pipe(
    prompt,
    guidance_scale=0.0,
    num_inference_steps=4,
    max_sequence_length=256,
    generator=generator
).images[0]

display(image)

stop = datetime.now()

In [ ]:
# Estimación de costo

seconds = (stop-start).total_seconds()
units_per_hour = 5.37
estimated_units = (5.37 / 3600) * seconds
estimated_cost = estimated_units * (9.99/100)
print(f"Esto tomó {seconds:.1f} segundos y costó un estimado de ${estimated_cost:.3f}")

# Pero hay un detalle: ¡pagas por todo el tiempo que el kernel está activo, no solo mientras calcula!
# Así que recuerda apagar un kernel de pago...